# 04 - Contributions

Who is putting the greenhouse gases there.

Notebook 02 built a complete panel: 206 countries x 24 years of national
greenhouse gas emissions, in kg CO2-eq. This notebook takes **one year - 2023,
the latest - and asks what share of the world's emissions came from each
country**, then aggregates the fourteen Pacific island countries and
territories into one figure.

That share is the number the first scene of the piece is built on. The ASR in
notebook 03 answers *did you overshoot your fair share*; this answers *how much
of it was you, right now*. They are different arguments and the piece uses both.

**One year, not a cumulative total.** A single year is what a map or a treemap
of the world can honestly show, and it needs no explanation of the window in
the label. It is also the framing that flatters the Pacific least - their share
of a 2023 snapshot is no smaller than their share of the 2000–2023 total, so
nobody can say the window was chosen. Change `YEAR` below to redo any other
year in the panel.

**Basis.** All greenhouse gases in CO2-eq, **excluding land use** - see the
caveats for why that exclusion is what the ratio requires rather than a
preference.

**Denominator.** The world total is the sum of the countries in the panel,
198 of them: those with both an emissions series and a World Bank population.

**Outputs**
- `data_viz/contributions.json` - per country and for the Pacific as a bloc, for the app
- `data_viz/contributions.csv` - the same table, for inspection

In [1]:
import json

import pandas as pd
import pycountry

from config import PACIFIC_ISO3, VIZ, YEARS

YEAR = max(YEARS)  # 2023

## 1. Load

`emissions.csv` is the merged table from notebook 02 - Pacific Data Hub for the
islands, Our World in Data for everyone else, one row per country-year.

In [2]:
emissions = pd.read_csv(VIZ / "emissions.csv")
countries = pd.read_csv(VIZ / "countries.csv")

year = emissions[emissions["year"] == YEAR].copy()

# Every country in the panel should appear exactly once in the chosen year.
# If one does not, the shares below are computed against a short denominator.
assert year["iso_code"].is_unique
assert len(year) == emissions["iso_code"].nunique(), (
    f"{len(year)} rows for {YEAR}, {emissions['iso_code'].nunique()} countries in the panel"
)

print(f"{len(year)} countries, {YEAR}")

198 countries, 2023


## 2. Contribution per country

Each country's emissions over the world's, in that one year. Per-capita comes
along for the ride - it is the same division the piece uses to argue that the
Pacific's smallness is not a virtue it chose.

In [3]:
GT = 1e12  # kg -> gigatonnes
MT = 1e9   # kg -> megatonnes

contrib = (
    year[["iso_code", "emissions_kg", "population", "emissions_t_per_capita"]]
    .assign(name=lambda d: d["iso_code"].map(countries.set_index("iso_code")["name"]),
            is_pacific=lambda d: d["iso_code"].isin(PACIFIC_ISO3))
)

world_kg = contrib["emissions_kg"].sum()
world_pop = contrib["population"].sum()

contrib["share_pct"] = contrib["emissions_kg"] / world_kg * 100
contrib["emissions_mt"] = contrib["emissions_kg"] / MT
contrib = (
    contrib.sort_values("emissions_kg", ascending=False)
    .assign(rank=lambda d: range(1, len(d) + 1))
    .reset_index(drop=True)
)

# The globe joins on ISO 3166-1 numeric, which is what world-atlas puts on its
# map features - no name matching, no hand-maintained lookup. pycountry has all
# 206; assert it rather than silently dropping a country to grey.
contrib["iso_n3"] = contrib["iso_code"].map(
    lambda a3: getattr(pycountry.countries.get(alpha_3=a3), "numeric", None)
)
assert contrib["iso_n3"].notna().all(), contrib.loc[contrib["iso_n3"].isna(), "iso_code"].tolist()

print(f"world, {YEAR}: {world_kg / GT:,.2f} Gt CO2-eq   "
      f"{world_pop / 1e9:.2f} bn people   {world_kg / world_pop / 1_000:.2f} t each")
contrib.head(15)[["rank", "name", "emissions_mt", "share_pct", "emissions_t_per_capita"]]

world, 2023: 43.26 Gt CO2-eq   8.03 bn people   5.39 t each


,rank,name,emissions_mt,share_pct,emissions_t_per_capita
0,1,China,13532.396,31.279798,9.754555
1,2,United States,5606.243,12.958692,16.627333
2,3,India,3431.968,7.932909,2.386510
3,4,Russia,2181.848,5.043288,15.170039
4,5,Indonesia,1093.872,2.528458,3.890152
5,6,Japan,1011.556,2.338187,8.123861
6,7,Iran,949.404,2.194524,10.478066
7,8,Saudi Arabia,789.141,1.824080,23.414749
8,9,Brazil,679.513,1.570677,3.218294
9,10,Canada,648.493,1.498976,16.178559


Two lines worth having as callouts: how far down the ranking you get before the
running total passes half the world, and how much the top twenty account for.

In [4]:
running = contrib["share_pct"].cumsum()

n_half = int((running < 50).sum() + 1)
print(f"{n_half} countries account for half of world emissions "
      f"({running.iloc[n_half - 1]:.1f}% by rank {n_half}: "
      f"{', '.join(contrib['name'].head(n_half))})")
print(f"top 20: {running.iloc[19]:.1f}%   |   bottom 100: {contrib['share_pct'].tail(100).sum():.2f}%")

3 countries account for half of world emissions (52.2% by rank 3: China, United States, India)
top 20: 80.3%   |   bottom 100: 1.58%


## 3. The Pacific

The fourteen island countries and territories, then the same fourteen treated
as one country - total emissions, total population, and the per-capita figure
that follows from both.

In [5]:
pacific = contrib[contrib["is_pacific"]]

pacific[["rank", "name", "emissions_mt", "share_pct", "emissions_t_per_capita", "population"]]

,rank,name,emissions_mt,share_pct,emissions_t_per_capita,population
123,124,Papua New Guinea,10.389635,0.024015,1.0,10389635.0
148,149,New Caledonia,5.246647,0.012127,18.1,289870.0
160,161,Fiji,2.680020,0.006195,2.9,924145.0
166,167,Palau,1.462478,0.003380,82.5,17727.0
172,173,French Polynesia,1.040137,0.002404,3.7,281118.0
179,180,Vanuatu,0.608777,0.001407,1.9,320409.0
181,182,Solomon Islands,0.560003,0.001294,0.7,800005.0
182,183,Samoa,0.498325,0.001152,2.3,216663.0
188,189,Tonga,0.282412,0.000653,2.7,104597.0
193,194,Kiribati,0.106024,0.000245,0.8,132530.0


In [6]:
pacific_kg = pacific["emissions_kg"].sum()
pacific_pop = pacific["population"].sum()

pacific_bloc = {
    "emissions_mt": pacific_kg / MT,
    "share_pct": pacific_kg / world_kg * 100,
    "population": pacific_pop,
    "population_share_pct": pacific_pop / world_pop * 100,
    "t_per_capita": pacific_kg / pacific_pop / 1_000,
    "world_t_per_capita": world_kg / world_pop / 1_000,
}

print(f"14 Pacific islands, {YEAR}: {pacific_bloc['emissions_mt']:,.1f} Mt CO2-eq")
print(f"  share of world emissions:  {pacific_bloc['share_pct']:.4f}%")
print(f"  share of world population: {pacific_bloc['population_share_pct']:.4f}%")
print(f"  per person: {pacific_bloc['t_per_capita']:.2f} t "
      f"vs {pacific_bloc['world_t_per_capita']:.2f} t worldwide "
      f"({pacific_bloc['world_t_per_capita'] / pacific_bloc['t_per_capita']:.1f}x)")

14 Pacific islands, 2023: 22.9 Mt CO2-eq
  share of world emissions:  0.0530%
  share of world population: 0.1700%
  per person: 1.68 t vs 5.39 t worldwide (3.2x)


### How small is that

A percentage with four leading zeros does not land. Two conversions that do:
how long the world takes to emit what the Pacific emits in a year, and where
the whole bloc would sit if it were a single country in the ranking.

In [7]:
seconds_per_year = 365.25 * 24 * 3600
world_kg_per_second = world_kg / seconds_per_year
world_equivalent_hours = pacific_kg / world_kg_per_second / 3600

bloc_rank = int((contrib["emissions_kg"] > pacific_kg).sum() + 1)
neighbour = contrib.iloc[bloc_rank - 1]  # the country the bloc would displace

print(f"The world emits a Pacific year in {world_equivalent_hours:.1f} hours.")
print(f"As one country the bloc would rank {bloc_rank} of {len(contrib) + 1}, "
      f"just above {neighbour['name']} ({neighbour['emissions_mt']:,.1f} Mt).")

The world emits a Pacific year in 4.6 hours.
As one country the bloc would rank 101 of 199, just above Croatia (21.6 Mt).


## 4. Write outputs

One JSON for the app - a meta block, the ranked country array, and the Pacific
bloc as its own object so a chart can draw it without re-aggregating. Values
are rounded at write time: the app has no use for sixteen significant figures
and the diff noise on re-runs is not worth it.

In [8]:
payload = {
    "meta": {
        "measure": f"greenhouse gas emissions, {YEAR}, CO2-eq",
        "year": int(YEAR),
        "world_gt": round(world_kg / GT, 2),
        "world_population": int(world_pop),
        "world_t_per_capita": round(pacific_bloc["world_t_per_capita"], 2),
        "countries": len(contrib),
        "denominator": f"sum of the {len(contrib)} countries in the panel; excludes ASM, GUM, MNP (no World Bank population) and eight countries with no land-use-free OWID series",
        "key": "iso_n3 is ISO 3166-1 numeric, zero-padded - the ids world-atlas puts on its map features",
        "sources": "Our World in Data (EDGAR / Global Carbon Project); Pacific Data Hub .Stat (SPC); World Bank population",
    },
    "countries": [
        {
            "iso_code": r.iso_code,
            "iso_n3": r.iso_n3,
            "name": r.name,
            "is_pacific": bool(r.is_pacific),
            "rank": int(r.rank),
            "emissions_mt": round(r.emissions_mt, 3),
            "share_pct": round(r.share_pct, 7),
            "t_per_capita": round(r.emissions_t_per_capita, 2),
            "population": int(r.population),
        }
        for r in contrib.itertuples()
    ],
    "pacific": {
        "countries": len(pacific),
        "rank_if_one_country": bloc_rank,
        "emissions_mt": round(pacific_bloc["emissions_mt"], 3),
        "share_pct": round(pacific_bloc["share_pct"], 7),
        "population": int(pacific_pop),
        "population_share_pct": round(pacific_bloc["population_share_pct"], 4),
        "t_per_capita": round(pacific_bloc["t_per_capita"], 2),
        "world_equivalent_hours": round(world_equivalent_hours, 1),
    },
}

(VIZ / "contributions.json").write_text(json.dumps(payload, indent=1))

contrib[[
    "rank", "iso_code", "iso_n3", "name", "is_pacific",
    "emissions_kg", "emissions_mt", "share_pct", "emissions_t_per_capita", "population",
]].to_csv(VIZ / "contributions.csv", index=False)

print(f"contributions.json  {(VIZ / 'contributions.json').stat().st_size / 1024:.0f} KB")
print(f"contributions.csv   {(VIZ / 'contributions.csv').stat().st_size / 1024:.0f} KB")

contributions.json  43 KB
contributions.csv   19 KB


## 5. Data quality flags

Notebook 02 carries the nearest observation across gaps in the OWID series so
a country with an interior hole is not dropped outright. For a country whose
series simply *ends* early that fill runs to the end of the window, and in a
single-year snapshot the repeated value **is** the datapoint - so it is worth
seeing which countries the chosen year is inferred for. Any run of identical
values at the tail is the signature.

This check earned its keep. It used to flag seven countries, and the cause was
not thin OWID coverage - it was notebook 02 filling *across country
boundaries*: `groupby(...).ffill()` hands back a plain Series, so the chained
`.bfill()` ran ungrouped and gave every country that opened with a gap the next
country's value alphabetically. Bermuda spent twenty-four years reporting
Bolivia's emissions, which is what put it at 1,795 t CO2-eq per resident.
Notebook 02 now fills inside `transform`, countries with no data at all are
dropped rather than invented, and this cell should print zero.


In [9]:
tail = (
    emissions.sort_values("year")
    .groupby("iso_code")["emissions_kg"]
    .apply(lambda s: (s == s.iloc[-1]).iloc[::-1].cummin().sum())
    .rename("flat_years")
)

flagged = (
    contrib.set_index("iso_code")
    .join(tail)
    .query("flat_years >= 3")
    .sort_values("emissions_mt", ascending=False)
)

print(f"{len(flagged)} countries whose {YEAR} value repeats 3+ years, "
      f"{flagged['share_pct'].sum():.2f}% of the world total, "
      f"{'none' if not flagged['is_pacific'].any() else 'SOME'} of them Pacific")
flagged[["rank", "name", "emissions_mt", "share_pct", "emissions_t_per_capita", "flat_years"]].head(12)

0 countries whose 2023 value repeats 3+ years, 0.00% of the world total, none of them Pacific


,rank,name,emissions_mt,share_pct,emissions_t_per_capita,flat_years
iso_code,,,,,,


## 6. Caveats

**Land use is excluded, on both sides.** These are all greenhouse gases in
CO2-eq - every Kyoto gas, not CO2 alone - but not forest clearing. That is
forced by the Pacific data: the Pacific Data Hub reports on that basis, so
matching OWID to it is what keeps one basis across all countries. (This was
originally justified by the AR6 budget's `include_afolu=False`; notebook 03
now uses a static carrying capacity, which is not built from an AFOLU setting,
so that argument no longer applies and the choice rests on data availability.) The Pacific Data Hub already
reports on this basis, so all three inputs now describe the same thing. It
costs the piece the deforestation story: here Brazil is the 9th largest
emitter rather than the 4th, and the DR Congo 67th rather than 11th.

**This is one year.** 2023 emissions, not a historical debt. The piece's
accountability argument needs the cumulative framing eventually - on a
since-1850 basis the United States and Europe move up and China down - but
that is a different chart with a different denominator, and it does not belong
on the same axis as this one.

**Territorial accounting.** Emissions counted where they physically happen. Two
Pacific entries look wrong at a glance and are not:

- **Palau**, 82.5 t CO2-eq per resident, the highest per-capita figure in the
  panel - half again as much as Qatar. It hosts several times its resident
  population in visitors, and the emissions are divided by residents.
- **New Caledonia**, 18.1 t, four times France. Nickel smelting, on an island
  of 290,000 people.

Both belong in the piece with that context attached - they are the Visitor
Paradox of Act 5 showing up early.

**Marshall Islands and Nauru look implausibly low** - around 0.1 t CO2-eq per
person, roughly a seventh of Kiribati's and a twentieth of Samoa's. These are
Pacific Data Hub figures and likely reflect partial sectoral coverage rather
than genuinely negligible emissions. They sit at the bottom of a ranking whose
bottom end the piece leans on, so verify against the national inventories
before quoting either one on its own. The bloc total is not sensitive to it:
both together are 0.005 Mt of 22.9.

**Shares are of a 198-country denominator.** Eight countries have no
land-use-free OWID series at all - Bermuda, Greenland, San Marino, Monaco,
Curacao, Sint Maarten, Eswatini, Palestine - and are dropped rather than
guessed at. American Samoa, Guam and the Northern Marianas remain out for the
older reason: Pacific Data Hub emissions but no World Bank population.
International aviation and shipping bunkers, which OWID reports separately and
which matter for small island states specifically, are in no country's total
and not in the world total either.